<a href="https://colab.research.google.com/github/saisudhanv/ML-Practice/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This notebook demonstrates different data preprocessing techniques for a COVID-19 related dataset. It covers:

# Data Loading and Exploration: Loading the covid_toy.csv dataset and performing initial checks like displaying the head, value counts for certain columns, and checking for missing values.
# Manual Data Preprocessing: Applying SimpleImputer for numerical features, OrdinalEncoder for ordinal categorical features, and OneHotEncoder for nominal categorical features, and then manually concatenating them.
# Automated Data Preprocessing with ColumnTransformer: Showing how to achieve the same preprocessing steps more efficiently and cleanly using ColumnTransformer from sklearn.compose.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/content/covid_toy.csv')

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df.head() # Cough is ordinal, gender and city is Nominal

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df['age'].value_counts()

,count
age,
65,4
19,4
34,4
51,3
64,3
20,3
42,3
69,3
82,3


In [ ]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [ ]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [ ]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [ ]:
X_train

,age,gender,fever,cough,city
34,74,Male,102.0,Mild,Mumbai
1,27,Male,100.0,Mild,Delhi
39,50,Female,103.0,Mild,Kolkata
22,71,Female,98.0,Strong,Kolkata
38,49,Female,101.0,Mild,Delhi
...,...,...,...,...,...
36,38,Female,101.0,Mild,Bangalore
3,31,Female,98.0,Mild,Kolkata
65,69,Female,102.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


Without Column Transformer

In [ ]:
# Adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])
# Also test data
X_test_fever = si.fit_transform(X_test[['fever']])


In [ ]:
X_train_fever.shape

(80, 1)

In [ ]:
# Ordinalcoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# Also test data
X_test_cough = oe.fit_transform(X_test[['cough']])

In [ ]:
X_train_cough.shape

(80, 1)

In [ ]:
# OneHot Encoding -> gender, city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

In [ ]:
X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# Also test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

In [ ]:
X_train_age.shape

(80, 1)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis = 1)
# Also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis = 1)

In [ ]:
X_train_transformed.shape

(80, 7)

Now using Column Transformer

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers = [
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [ ]:
transformer.fit_transform(X_train).shape

(80, 7)

In [ ]:
transformer.fit_transform(X_test).shape

(20, 7)